# Bloch equations and the spin echo

*medical · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/bloch_spin_echo)

The Bloch equation is a cross product, $\varepsilon_{ijk} M_j \Omega_k$, plus relaxation. Hundreds of isochromats with slightly different off-resonance frequencies $\Delta\omega_n$ are stepped together in one batched RK4 update: after a 90° pulse they fan out and the signal dies with $T_2^*$, then a 180° pulse at $TE/2$ makes them refocus into an echo at $TE$ whose height decays only with the true $T_2$.

**Vector form**

$$\frac{d\mathbf M}{dt} = \gamma\,\mathbf M\times\mathbf B - \frac{M_x\hat{\mathbf x} + M_y\hat{\mathbf y}}{T_2} - \frac{(M_z - M_0)\,\hat{\mathbf z}}{T_1},\quad s = \langle M_x + iM_y\rangle$$

**Index (tensor) form**

$$\dot M_{ni} = \varepsilon_{ijk}\,M_{nj}\,\Omega_{nk} - R_{ij}\,(M_{nj} - M^0_j),\qquad \Omega_{nk} = (\omega_1\cos\phi,\ \omega_1\sin\phi,\ \Delta\omega_n),\qquad s = \tfrac1N\textstyle\sum_n (M_{nx} + iM_{ny})$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $M_{ni}$ | `M` | (N, 3) | magnetisation of isochromat $n$, component $i \in \{x, y, z\}$, in the rotating frame |
| $\varepsilon_{ijk}$ | `levi_civita()` | (3, 3, 3) | Levi-Civita symbol: the cross product, exactly as in the Lorentz force |
| $\Omega_{nk}$ | `omega(dw, w1, phase)` | (N, 3) | $\gamma\mathbf B$ in the rotating frame: RF field $\omega_1$ in the $xy$-plane during a pulse, off-resonance $\Delta\omega_n$ along $z$ |
| $\varepsilon_{ijk} M_{nj}\Omega_{nk}$ | `einsum('ijk,nj,nk->ni', eps, M, Om)` | (N, 3) | repeated $j, k$ ⇒ summed: precession $\mathbf M\times\boldsymbol\Omega$ of every isochromat |
| $R_{ij}$ | `relaxation(T1, T2)` | (3, 3) | $\mathrm{diag}(1/T_2, 1/T_2, 1/T_1)$; contracted with $M_{nj} - M^0_j$ over $j$ |
| $\Delta\omega_n$ | `off_resonance(N, T2p, kind)` | (N,) | spread of precession frequencies from field inhomogeneity, width $1/T_2^{\prime}$ |
| $s$ | `signal(M)` | (T,) complex | what the receive coil sees: the mean transverse magnetisation, summed over $n$ |

Times are in ms and frequencies in rad/ms. Each isochromat alone keeps $\lvert\mathbf M\rvert$ fixed apart from relaxation, but the ensemble average loses phase coherence: with a Lorentzian spread the free induction decay follows $e^{-t/T_2^*}$ with $1/T_2^* = 1/T_2 + 1/T_2^{\prime}$. The 180° pulse mirrors every phase, so spins that ran ahead are now behind and all line up again at $TE$; this cancels the static inhomogeneity but not the random $T_2$ loss, so the echo heights trace $e^{-TE/T_2}$. Several refocusing pulses (a CPMG train, 180° about $y$) give a row of echoes that maps out $T_2$. Pulses are short rectangular "hard" pulses run through the same Bloch equation; spins far off resonance are refocused imperfectly. Values are illustrative, for education only, not for clinical use. Try the Gaussian spread, a longer $T_2^{\prime}$ or a single echo.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/bloch_spin_echo/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T1, T2, TE, n_echoes = params["T1"], params["T2"], params["TE"], params["n_echoes"]
    eps = tensors.levi_civita(dev)
    R, M0 = tensors.relaxation(T1, T2, dev)
    dw = tensors.off_resonance(params["n_spins"], params["t2_prime"], params["spread"], dev)
    M_init = tensors.equilibrium_state(params["n_spins"], dev)
    pulse_len = 0.2  # ms, a short "hard" pulse
    pulses = tensors.pulse_list(TE, n_echoes, pulse_len)
    t90 = pulse_len / 2

    t_max = t90 + (n_echoes + 0.4) * TE
    times, M = solve.solve(M_init, dw, eps, R, M0, pulses, t_max)

    t2_star = solve.fid_decay_time(times, M, t90, TE)
    print(f"T2 = {T2:.1f} ms,  T2' = {params['t2_prime']:.1f} ms,  measured T2* (FID to 1/e) = {t2_star:.2f} ms")
    t_echo, height = solve.echo_peaks(times, M, TE, n_echoes, t90)
    for k, (te, h) in enumerate(zip(t_echo.tolist(), height.tolist()), start=1):
        expected = math.exp(-k * TE / T2)
        print(f"echo {k}: t = {te - t90:6.2f} ms (expected {k * TE:.1f})  |s| = {h:.4f}  e^(-t/T2) = {expected:.4f}")
    return plot.animate(times, M, dw, pulses, T2, t2_star, t90, out_dir / "bloch_spin_echo.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/bloch_spin_echo/equation.py
r"""Physics → tensor translation.

The Bloch equation for an ensemble of isochromats n (spins that see the same field), in the frame rotating
at the Larmor frequency, with the field written as an angular frequency \Omega = \gamma B:

    dM_{ni}/dt = \varepsilon_{ijk} M_{nj} \Omega_{nk}  -  R_{ij} (M_{nj} - M^0_j)        (j, k summed)

    \Omega_{nk} = (\omega_1 \cos\phi,\ \omega_1 \sin\phi,\ \Delta\omega_n)    RF pulse in x/y, off-resonance along z
    R_{ij}      = diag(1/T_2, 1/T_2, 1/T_1),     M^0_j = (0, 0, M_0)

The first term is the cross product M × Ω (the same \varepsilon_{ijk} contraction as the Lorentz force);
the second pulls M_z back to M_0 with T_1 and kills M_x, M_y with T_2. The coil measures the average
transverse magnetisation of all isochromats, a sum over n:

    s = (1/N) \sum_n (M_{nx} + i M_{ny})
"""

import math

import torch


def omega(dw: torch.Tensor, w1: float, phase: float) -> torch.Tensor:
    r"""\Omega_{nk} = (\omega_1\cos\phi, \omega_1\sin\phi, \Delta\omega_n) in rad/ms.   dw: (N,) → (N, 3)"""
    x = torch.full_like(dw, w1 * math.cos(phase))
    y = torch.full_like(dw, w1 * math.sin(phase))
    return torch.stack([x, y, dw], dim=-1)


def bloch_rhs(M: torch.Tensor, Om: torch.Tensor, eps: torch.Tensor, R: torch.Tensor, M0: torch.Tensor) -> torch.Tensor:
    r"""dM_{ni}/dt = \varepsilon_{ijk} M_{nj} \Omega_{nk} - R_{ij}(M_{nj} - M^0_j).   M, Om: (N, 3) → (N, 3)"""
    precession = torch.einsum("ijk,nj,nk->ni", eps, M, Om)  # j, k summed: M × Ω for every isochromat
    relaxation = torch.einsum("ij,nj->ni", R, M - M0)  # j summed: diagonal relaxation rates
    return precession - relaxation


def signal(M: torch.Tensor) -> torch.Tensor:
    r"""s = (1/N) \sum_n (M_{nx} + i M_{ny}).   M: (..., N, 3) → (...,) complex"""
    return torch.complex(M[..., 0], M[..., 1]).mean(dim=-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/bloch_spin_echo/tensors.py
"""Build the concrete tensors: Levi-Civita symbol, relaxation matrix, off-resonance spread and the pulse list."""

import math

import torch


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_{ijk}: +1 for even permutations of (x, y, z), −1 for odd ones, 0 otherwise. Shape (3, 3, 3)."""
    eps = torch.zeros(3, 3, 3, dtype=torch.float64, device=device)
    for i, j, k in [(0, 1, 2), (1, 2, 0), (2, 0, 1)]:
        eps[i, j, k] = 1.0
        eps[j, i, k] = -1.0
    return eps


def relaxation(T1: float, T2: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """R_{ij} = diag(1/T2, 1/T2, 1/T1) (3, 3) and the equilibrium M^0_j = (0, 0, 1) (3,). Times in ms."""
    R = torch.diag(torch.tensor([1 / T2, 1 / T2, 1 / T1], dtype=torch.float64, device=device))
    M0 = torch.tensor([0.0, 0.0, 1.0], dtype=torch.float64, device=device)
    return R, M0


def off_resonance(n: int, t2_prime: float, kind: str, device: torch.device) -> torch.Tensor:
    """Δω_n (rad/ms) for n isochromats, spread by field inhomogeneity with width 1/T2'. Shape (N,).

    The spins sit at evenly spaced quantiles u_n of the line shape (deterministic, no random numbers):
    Lorentzian Δω = tan(π(u − ½))/T2' gives an exponential FID e^{−t/T2'}; Gaussian gives e^{−t²/2T2'²}.
    """
    u = (torch.arange(n, dtype=torch.float64, device=device) + 0.5) / n  # (N,) in (0, 1)
    width = 1.0 / t2_prime
    if kind == "gaussian":
        return width * math.sqrt(2.0) * torch.erfinv(2 * u - 1)
    u = 0.01 + 0.98 * u  # clip the far Lorentzian tails
    return width * torch.tan(math.pi * (u - 0.5))


def equilibrium_state(n: int, device: torch.device) -> torch.Tensor:
    """M_{ni} = (0, 0, 1) for every isochromat: all magnetisation along the main field. Shape (N, 3)."""
    M = torch.zeros(n, 3, dtype=torch.float64, device=device)
    M[:, 2] = 1.0
    return M


def pulse_list(TE: float, n_echoes: int, duration: float) -> list[tuple[float, float, float, float]]:
    """(start, length, flip angle, phase) of every RF pulse, times in ms.

    A 90° pulse about x at t = 0, then refocusing 180° pulses about y (CPMG) centred at TE/2, 3TE/2, ...
    measured from the centre of the 90° pulse, so echo k forms at k·TE after it.
    """
    pulses = [(0.0, duration, math.pi / 2, 0.0)]
    t90 = duration / 2  # centre of the excitation
    for k in range(n_echoes):
        centre = t90 + (k + 0.5) * TE
        pulses.append((centre - duration / 2, duration, math.pi, math.pi / 2))
    return pulses


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/bloch_spin_echo/solve.py
"""Integrate the Bloch equation for all isochromats at once with RK4, switching the RF field on during pulses."""

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import bloch_rhs, omega, signal


def rf_at(t: float, pulses: list[tuple[float, float, float, float]]) -> tuple[float, float]:
    """(ω1, phase) of the RF field at time t: ω1 = flip / length inside a pulse, 0 outside."""
    for start, length, flip, phase in pulses:
        if start <= t < start + length:
            return flip / length, phase
    return 0.0, 0.0


def solve(
    M_init: torch.Tensor,
    dw: torch.Tensor,
    eps: torch.Tensor,
    R: torch.Tensor,
    M0: torch.Tensor,
    pulses: list[tuple[float, float, float, float]],
    t_max: float,
    dt: float = 0.05,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), M (T, N, 3)).

    The field Ω_{nk} is held constant over each step (taken at the step's midpoint), so a pulse of length τ
    rotates the magnetisation by ω1·τ = the flip angle. All N isochromats advance in one batched RK4 step.
    """
    n_steps = int(round(t_max / dt))
    states = [M_init]
    M = M_init
    for n in steps(n_steps, "Bloch steps"):
        w1, phase = rf_at((n + 0.5) * dt, pulses)
        Om = omega(dw, w1, phase)  # (N, 3)
        M = rk4_step(lambda y, t, Om=Om: bloch_rhs(y, Om, eps, R, M0), M, n * dt, dt)  # Ω fixed for this step
        states.append(M)
    times = dt * torch.arange(n_steps + 1, dtype=torch.float64, device=M_init.device)
    return times, torch.stack(states)


def echo_peaks(
    times: torch.Tensor, M: torch.Tensor, TE: float, n_echoes: int, t90: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Time (K,) and height (K,) of the largest |s| within ±TE/4 of each expected echo t90 + k·TE."""
    mag = signal(M).abs()  # (T,)
    t_peak, height = [], []
    for k in range(1, n_echoes + 1):
        window = (times - (t90 + k * TE)).abs() < TE / 4
        i = torch.argmax(torch.where(window, mag, torch.zeros_like(mag)))
        t_peak.append(times[i])
        height.append(mag[i])
    return torch.stack(t_peak), torch.stack(height)


def fid_decay_time(times: torch.Tensor, M: torch.Tensor, t90: float, TE: float) -> float:
    """First time after the 90° pulse when |s| falls below 1/e: the measured T2* (before the first 180°)."""
    mag = signal(M).abs()
    after = (times > t90) & (times < t90 + TE / 2)
    below = after & (mag < torch.exp(torch.tensor(-1.0, dtype=mag.dtype)))
    if not below.any():
        return float("nan")
    return float(times[below.int().argmax()] - t90)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/bloch_spin_echo/plot.py
"""Animate isochromat arrows fanning out and refocusing in the transverse plane, next to the signal |s|(t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import signal


def animate(
    times: torch.Tensor,
    M: torch.Tensor,
    dw: torch.Tensor,
    pulses: list[tuple[float, float, float, float]],
    T2: float,
    t2_star: float,
    t90: float,
    path: Path,
) -> Path:
    t = times.cpu().numpy()
    mag = signal(M).abs().cpu().numpy()  # (T,)
    n = M.shape[1]
    shown = np.linspace(0, n - 1, min(n, 31)).round().astype(int)  # a few isochromats to draw
    Mxy = M[:, shown, :2].cpu().numpy()  # (T, n_shown, 2)
    net = signal(M).cpu().numpy()  # (T,) complex average
    colours = plt.cm.coolwarm(np.linspace(0, 1, len(shown)))  # blue = slower, red = faster than resonance

    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(7.6, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
    ax_p.set_xlim(-1.1, 1.1)
    ax_p.set_ylim(-1.1, 1.1)
    ax_p.set_aspect("equal")
    ax_p.set_xlabel("$M_x$", labelpad=0)
    ax_p.set_ylabel("$M_y$", labelpad=0)
    ax_p.add_patch(plt.Circle((0, 0), 1, fill=False, color="#9ca3af", lw=0.8, ls=":"))
    arrows = [ax_p.plot([], [], color=c, lw=1.2, alpha=0.85)[0] for c in colours]
    (net_arrow,) = ax_p.plot([], [], color="#111827", lw=3)
    title = ax_p.set_title("")

    for start, length, flip, _ in pulses:
        ax_s.axvspan(start, start + length, color="#f59e0b", alpha=0.5, lw=0)
        ax_s.text(start + length / 2, 1.05, f"{np.degrees(flip):.0f}°", ha="center", fontsize=7, color="#b45309")
    tt = t[t >= t90]
    ax_s.plot(tt, np.exp(-(tt - t90) / T2), color="#16a34a", ls="--", lw=1, label="$e^{-t/T_2}$ (echo envelope)")
    if np.isfinite(t2_star):
        ax_s.plot(tt, np.exp(-(tt - t90) / t2_star), color="#9333ea", ls=":", lw=1.2, label="$e^{-t/T_2^*}$ (FID)")
    (curve,) = ax_s.plot([], [], color="#2563eb", lw=1.6, label="$|s(t)|$")
    ax_s.set_xlim(t[0], t[-1])
    ax_s.set_ylim(0, 1.12)
    ax_s.set_xlabel("t (ms)", labelpad=0)
    ax_s.set_ylabel("net transverse signal")
    ax_s.legend(loc="center right", fontsize=7, framealpha=0.9)

    def update(step: int):
        for arrow, (mx, my) in zip(arrows, Mxy[step]):
            arrow.set_data([0, mx], [0, my])
        net_arrow.set_data([0, net[step].real], [0, net[step].imag])
        curve.set_data(t[: step + 1], mag[: step + 1])
        title.set_text(f"t = {t[step]:5.1f} ms   |s| = {mag[step]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 96), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T1': 800.0,
    'T2': 80.0,
    't2_prime': 4.0,
    'TE': 24.0,
    'n_echoes': 4,
    'n_spins': 400,
    'spread': 'lorentzian'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.bloch_spin_echo")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.bloch_spin_echo.main")
problem_dir = Path("problems/medical/bloch_spin_echo")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
